In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_E.xpt"
csv_file = DATA_DIR / "GHB_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 GHB_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if df.shape != (6917, 2):

    raise ValueError(
        f"Unexpected GHB_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GHB_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: GHB_E structure = "
    "6,917 rows x 2 columns."
)


# ============================================================
# 4. FIELD TYPES
# ============================================================

#
# SEQN:
# identifier -> clean integer representation
#
# LBXGH:
# Glycohemoglobin / HbA1c %
# REAL LAB MEASUREMENT
#
# KEEP LBXGH AS SOURCE-DECODED FLOAT.
#
# DO NOT:
#
# - Int64 LBXGH
# - round LBXGH
# - trim legitimate .0
# - recalculate LBXGH
# - use float_format
#

INTEGER_COLUMNS = [
    "SEQN"
]


MEASUREMENT_COLUMNS = [
    "LBXGH"
]


print(
    "\nInteger identifier fields:",
    INTEGER_COLUMNS
)

print(
    "Lab measurement fields preserved AS IS:",
    MEASUREMENT_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT value(s) found. "
        "Do NOT automatically replace them with zero."
    )


print(
    "PASS: GHB_E contains no tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "Unexpected source zero(s) found."
    )


print(
    "PASS: Original GHB_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


seqn_missing = int(
    seqn.isna()
    .sum()
)


seqn_duplicates = int(
    seqn.duplicated()
    .sum()
)


seqn_min = seqn.min()
seqn_max = seqn.max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:

    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. LBXGH CDC VALID/MISSING/RANGE CHECK
# ============================================================

#
# CDC 2007-2008:
#
# LBXGH Glycohemoglobin (%)
#
# Range   = 2 to 15.2
# Valid   = 6,427
# Missing = 490
#

s = df["LBXGH"]


lbxgh_valid = int(
    s.notna()
    .sum()
)


lbxgh_missing = int(
    s.isna()
    .sum()
)


lbxgh_min = s.min()
lbxgh_max = s.max()


print(
    "\n--- LBXGH CDC CHECK ---"
)

print(
    "Valid:",
    f"{lbxgh_valid:,}"
)

print(
    "Missing:",
    f"{lbxgh_missing:,}"
)

print(
    "Minimum:",
    lbxgh_min
)

print(
    "Maximum:",
    lbxgh_max
)


if lbxgh_valid != 6427:

    raise ValueError(
        f"LBXGH valid count differs: "
        f"{lbxgh_valid:,}"
    )


if lbxgh_missing != 490:

    raise ValueError(
        f"LBXGH missing count differs: "
        f"{lbxgh_missing:,}"
    )


if not np.isclose(
    lbxgh_min,
    2.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected LBXGH minimum: {lbxgh_min}"
    )


if not np.isclose(
    lbxgh_max,
    15.2,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected LBXGH maximum: {lbxgh_max}"
    )


print(
    "PASS: LBXGH matches CDC "
    "(6,427 valid, 490 missing, range 2-15.2)."
)


# ============================================================
# 9. LBXGH GENUINE DECIMAL CHECK
# ============================================================

#
# LBXGH is a genuine lab measurement.
#
# It MUST NOT be converted to Int64.
#

values = (
    df["LBXGH"]
    .dropna()
    .to_numpy()
)


fractional_mask = ~np.isclose(
    values,
    np.round(values),
    rtol=0,
    atol=1e-12
)


lbxgh_fractional = int(
    fractional_mask.sum()
)


lbxgh_whole = int(
    (~fractional_mask).sum()
)


print(
    "\n--- LBXGH DECIMAL CHECK ---"
)

print(
    "Fractional values:",
    f"{lbxgh_fractional:,}"
)

print(
    "Whole-number values:",
    f"{lbxgh_whole:,}"
)


if lbxgh_fractional != 5834:

    raise ValueError(
        "Unexpected LBXGH fractional-value count."
    )


if lbxgh_whole != 593:

    raise ValueError(
        "Unexpected LBXGH whole-number count."
    )


print(
    "PASS: LBXGH genuine decimals validated."
)


# ============================================================
# 10. SOURCE .0 REPRESENTATION CHECK
# ============================================================

#
# The 593 mathematically whole LBXGH values are
# still genuine LAB MEASUREMENTS.
#
# Example:
#
# 7.0 must remain "7.0"
#
# NOT:
#
# 7
#

source_lbxgh_dot_zero = sum(
    str(
        float(value)
    ).endswith(".0")
    for value in df["LBXGH"].dropna()
)


print(
    "\nSource LBXGH values ending '.0':",
    source_lbxgh_dot_zero
)


if source_lbxgh_dot_zero != 593:

    raise ValueError(
        "Unexpected source LBXGH .0 count."
    )


print(
    "PASS: 593 legitimate LBXGH .0 "
    "values identified."
)


# ============================================================
# 11. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# IMPORTANT:
#
# GHB_E requires NO numeric corrections.
#
# CDC says:
#
# - no derived variables
# - no fill values
# - no recoding
#
# So expected_df equals the original released XPT.
#

expected_df = source_df.copy()


# ============================================================
# 12. CLEAN SEQN ONLY
# ============================================================

#
# SEQN:
#
# 41475.0 -> 41475
#
#
# LBXGH:
#
# 5.7 -> 5.7
# 7.0 -> 7.0
#
# DO NOT Int64 LBXGH.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to clean integer formatting."
)

print(
    "PASS: LBXGH left as source-decoded float."
)


# ============================================================
# 13. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


# SEQN numeric equality
current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    expected_df["SEQN"].to_numpy(),
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN numeric values changed."
    )


# LBXGH exact equality
if not np.allclose(
    expected_df["LBXGH"].to_numpy(),
    df["LBXGH"].to_numpy(),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH values changed before export."
    )


print(
    "PASS: No underlying numeric value changed."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# recalculation
# decimal truncation
#
# Missing SAS values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 15. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 16. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 17. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: SEQN and LBXGH numerically "
    "match original XPT exactly."
)


# ============================================================
# 18. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 19. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 6917:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 20. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# SEQN:
#
# expected:
# str(int(source_value))
#
#
# LBXGH:
#
# expected:
# str(float(source_value))
#
# This preserves:
#
# 5.7 -> "5.7"
# 7.0 -> "7.0"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    # -----------------------------
    # SEQN
    # -----------------------------

    seqn_value = (
        expected_df.iloc[i]["SEQN"]
    )


    expected_seqn_text = (
        str(
            int(seqn_value)
        )
    )


    if (
        row["SEQN"]
        !=
        expected_seqn_text
    ):

        raw_mismatches += 1


        if len(raw_examples) < 20:

            raw_examples.append(
                {
                    "csv_row": i + 2,
                    "column": "SEQN",
                    "expected": expected_seqn_text,
                    "actual": row["SEQN"]
                }
            )


    # -----------------------------
    # LBXGH
    # -----------------------------

    lbxgh_value = (
        expected_df.iloc[i]["LBXGH"]
    )


    expected_lbxgh_text = (
        ""
        if pd.isna(lbxgh_value)
        else str(
            float(lbxgh_value)
        )
    )


    if (
        row["LBXGH"]
        !=
        expected_lbxgh_text
    ):

        raw_mismatches += 1


        if len(raw_examples) < 20:

            raw_examples.append(
                {
                    "csv_row": i + 2,
                    "column": "LBXGH",
                    "expected": expected_lbxgh_text,
                    "actual": row["LBXGH"]
                }
            )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "GHB_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV tokens match expected "
    "source representation exactly."
)


# ============================================================
# 21. SEQN .0 CHECK
# ============================================================

seqn_dot_zero_count = sum(
    row["SEQN"].endswith(".0")
    for row in raw_rows
    if row["SEQN"] != ""
)


print(
    "\nUnwanted SEQN '.0' values:",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains .0 formatting."
    )


print(
    "PASS: SEQN has no unwanted .0."
)


# ============================================================
# 22. LBXGH LEGITIMATE .0 CHECK
# ============================================================

#
# IMPORTANT:
#
# Do NOT remove .0 from LBXGH.
#
# 7.0 is a real laboratory measurement
# and must remain 7.0.
#

final_lbxgh_dot_zero = sum(
    row["LBXGH"].endswith(".0")
    for row in raw_rows
    if row["LBXGH"] != ""
)


print(
    "Legitimate LBXGH '.0' values:",
    final_lbxgh_dot_zero
)


if (
    final_lbxgh_dot_zero
    !=
    source_lbxgh_dot_zero
):

    raise ValueError(
        "LBXGH legitimate .0 formatting changed."
    )


if final_lbxgh_dot_zero != 593:

    raise ValueError(
        "Unexpected LBXGH .0 count."
    )


print(
    "PASS: All 593 legitimate LBXGH .0 "
    "values preserved."
)


# ============================================================
# 23. STRICT LBXGH RAW MEASUREMENT CHECK
# ============================================================

lbxgh_raw_mismatches = 0
lbxgh_examples = []


for i, row in enumerate(
    raw_rows
):

    source_value = (
        expected_df.iloc[i]["LBXGH"]
    )


    expected_text = (
        ""
        if pd.isna(source_value)
        else str(
            float(source_value)
        )
    )


    actual_text = (
        row["LBXGH"]
    )


    if actual_text != expected_text:

        lbxgh_raw_mismatches += 1


        if len(
            lbxgh_examples
        ) < 20:

            lbxgh_examples.append(
                {
                    "csv_row": i + 2,
                    "expected": expected_text,
                    "actual": actual_text
                }
            )


print(
    "\nLBXGH raw measurement differences:",
    lbxgh_raw_mismatches
)


if lbxgh_raw_mismatches != 0:

    print(
        lbxgh_examples
    )

    raise ValueError(
        "LBXGH decimal representation changed."
    )


print(
    "PASS: Every LBXGH value preserved AS IS."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 26. FINAL LBXGH CDC CHECK
# ============================================================

final_lbxgh = (
    csv_df["LBXGH"]
)


if int(
    final_lbxgh.notna()
    .sum()
) != 6427:

    raise ValueError(
        "Final LBXGH valid count changed."
    )


if int(
    final_lbxgh.isna()
    .sum()
) != 490:

    raise ValueError(
        "Final LBXGH missing count changed."
    )


if not np.isclose(
    final_lbxgh.min(),
    2.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Final LBXGH minimum changed."
    )


if not np.isclose(
    final_lbxgh.max(),
    15.2,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Final LBXGH maximum changed."
    )


print(
    "\nPASS: Final LBXGH still matches CDC:"
)

print(
    "      Valid   = 6,427"
)

print(
    "      Missing = 490"
)

print(
    "      Range   = 2.0-15.2"
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GHB_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts in source:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)


print(
    "\nLBXGH valid:",
    f"{lbxgh_valid:,}"
)

print(
    "LBXGH missing:",
    f"{lbxgh_missing:,}"
)

print(
    "LBXGH range:",
    "2.0-15.2"
)

print(
    "LBXGH fractional values:",
    f"{lbxgh_fractional:,}"
)

print(
    "LBXGH legitimate .0 values:",
    f"{final_lbxgh_dot_zero:,}"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "LBXGH raw differences:",
    lbxgh_raw_mismatches
)

print(
    "Unwanted SEQN .0:",
    seqn_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "GHB_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 GHB_E XPT -> CSV
Rows: 6,917
Columns: 2
PASS: GHB_E structure = 6,917 rows x 2 columns.

Integer identifier fields: ['SEQN']
Lab measurement fields preserved AS IS: ['LBXGH']

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: GHB_E contains no tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original GHB_E contains no numeric zeros.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 41475.0
Maximum: 51623.0
PASS: SEQN validated.

--- LBXGH CDC CHECK ---
Valid: 6,427
Missing: 490
Minimum: 2.0
Maximum: 15.2
PASS: LBXGH matches CDC (6,427 valid, 490 missing, range 2-15.2).

--- LBXGH DECIMAL CHECK ---
Fractional values: 5,834
Whole-number values: 593
PASS: LBXGH genuine decimals validated.

Source LBXGH values ending '.0': 593
PASS: 593 legitimate LBXGH .0 values identified.

PASS: SEQN converted to clean integer formatting.
PASS: LBXGH left as source-decoded float.

--- PRE-EXPORT NUMERIC FIDELITY ---
PASS: No underlying numeric value changed.